# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

To understand the structure of the dataset, let's list all record sets and their fields. [All entities are referenced by their `@id`.]

In [ ]:
# List all record sets
record_sets = list(metadata.record_sets())
print(f"Record sets (total={len(record_sets)}):\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name} (ID: {rs.id})")
    fields = list(rs.fields())
    print("  Fields:")
    for field in fields:
        columns = list(field.columns())
        col_ids = [col.id for col in columns]
        print(f"    - Field: {field.name} (@id: {field.id}), Columns: {col_ids}")
    print()
# Show a sample of records in each RecordSet
for rs in record_sets:
    print(f"Sample record from RecordSet '{rs.name}' (@id: {rs.id}):")
    sample = next(dataset.records(record_set=rs.id), None)
    if sample is not None:
        print(sample)
    else:
        print("  (No sample record available)")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We select all record sets available. For demonstration, we load all into Pandas DataFrames using their `@id`.

In [ ]:
# Extract data from each record set
all_record_set_ids = [rs.id for rs in metadata.record_sets()]
dataframes = {}

for record_set_id in all_record_set_ids:
    # Fetch all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet '@id': {record_set_id}")
    else:
        print(f"No records found for RecordSet '@id': {record_set_id}")
        dataframes[record_set_id] = pd.DataFrame()

# Select first RecordSet as main example
if len(all_record_set_ids) > 0:
    main_rs_id = all_record_set_ids[0]
    print("\nColumns in main RecordSet:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

We first list numeric fields, select one for filtering, and show the effect. [Refer to all fields by their `@id`.]

In [ ]:
import numpy as np

# Just for the main RecordSet
main_df = dataframes[main_rs_id]
main_rs_obj = [rs for rs in metadata.record_sets() if rs.id == main_rs_id][0]
field_objs = list(main_rs_obj.fields())

# Identify numeric fields by Croissant dataType, collect their @id
numeric_fields = []
for field in field_objs:
    dtype = getattr(field, 'data_type', None)
    if dtype in ['Float', 'Number', 'Integer']:
        numeric_fields.append(field)

print("Numeric fields in main RecordSet:")
for nf in numeric_fields:
    print(f"- {nf.name} (@id: {nf.id}) | type: {nf.data_type}")

# Choose the first numeric field for demonstration
if numeric_fields:
    selected_numeric_field_id = numeric_fields[0].id
    col_name = selected_numeric_field_id  # Croissant maps fields by @id
    # Check if column exists in DataFrame
    if col_name not in main_df.columns:
        print(f"Column '{col_name}' not found in DataFrame.")
    else:
        threshold = main_df[col_name].astype(float).mean() # Use mean as threshold
        filtered_df = main_df[main_df[col_name].astype(float) > threshold].copy()
        print(f"Filtered records where {col_name} > {threshold:.2f} (field @id: {selected_numeric_field_id}): {len(filtered_df)} rows")

        # Normalization
        filtered_df[f"{col_name}_normalized"] = (filtered_df[col_name].astype(float) - filtered_df[col_name].astype(float).mean()) / filtered_df[col_name].astype(float).std()
        print(f"\nFirst 5 normalized records for {col_name}:")
        display(filtered_df[[col_name, f"{col_name}_normalized"]].head())

        # Pick a categorical/group field if available
        group_fields = [f for f in field_objs if getattr(f, 'data_type', '') in ['Text', None] and f.id != selected_numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0].id
            print(f"\nGrouping by field '@id': {group_field_id}")
            if group_field_id in filtered_df.columns:
                grouped = filtered_df.groupby(group_field_id)[col_name].mean()
                print(grouped.head())
else:
    print("No numeric fields found in main RecordSet.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we will plot the distribution of the selected numeric field and, if a group is available, show group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    col_name = selected_numeric_field_id
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[col_name].astype(float).dropna(), bins=30, kde=True)
    plt.title(f"Distribution of field '@id': {col_name}")
    plt.xlabel(col_name)
    plt.ylabel("Frequency")
    plt.show()

    # If group field exists
    if 'group_field_id' in locals():
        plt.figure(figsize=(10,6))
        # Limit to 20 groups for clarity
        means = main_df.groupby(group_field_id)[col_name].mean().sort_values().head(20)
        sns.barplot(y=means.index, x=means.values)
        plt.title(f"Mean of '{col_name}' by '{group_field_id}' (first 20)")
        plt.xlabel(f"Mean of {col_name}")
        plt.ylabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
- Loaded metadata and records dynamically from the FAIR^2 protein dataset (Croissant schema)
- Explored available record sets and their field structures (all by `@id`)
- Loaded tabular data, filtered and normalized a numeric field
- Optionally grouped by a categorical field and visualized numeric distributions

This approach can be extended to more advanced analyses as needed.